# Day 10: End-to-End Orchestration, Observability & Verification

This notebook demonstrates a complete E2E run of a synthetic patient from raw JSON -> Bronze -> Tokenization -> Silver OMOP -> DQ -> Gold readmission metric.

In [ ]:
import os
import hashlib
import duckdb
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, udf, expr
from pyspark.sql.types import StringType

os.environ['SPARK_VERSION'] = '3.5'
os.environ['EHDIP_HASH_SALT'] = 'test_salt_e2e'

spark = SparkSession.builder \
    .appName("Day10_E2E_Pipeline") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark Initialized for E2E Pipeline Walkthrough.")


### Step 1: Raw JSON to Bronze

In [ ]:
# Synthetic HL7/FHIR JSON
raw_data = [
    ("P100", "111-22-3333", "MRN001", "1985-06-15", "32020", "2023-01-01", "Initial Admit"),
    ("P100", "111-22-3333", "MRN001", "1985-06-15", "32020", "2023-01-25", "Readmit < 30 days"),
    (None, None, None, None, "-1", "2023-02-01", "Bad Record") # Intentional bad record
]

columns = ["patient_id", "ssn", "mrn", "birth_date", "condition_code", "condition_start_date", "notes"]
bronze_df = spark.createDataFrame(raw_data, columns)

print("Bronze Layer (Raw Payload):")
bronze_df.show(truncate=False)


### Step 2: Tokenization & De-ID (Silver prep)

In [ ]:
def _fpe_mask(value):
    if not value:
        return value
    salt = os.environ.get("EHDIP_HASH_SALT", "default")
    return hashlib.sha256(f"{value}{salt}".encode('utf-8')).hexdigest()[:10] # Truncated for readability

fpe_udf = udf(_fpe_mask, StringType())

deid_df = bronze_df \
    .withColumn("ssn_masked", fpe_udf(col("ssn"))) \
    .withColumn("mrn_masked", fpe_udf(col("mrn"))) \
    .withColumn("date_shift_days", expr("pmod(conv(substr(md5(patient_id), 1, 15), 16, 10), 61) - 30")) \
    .withColumn("birth_date_shifted", expr("date_add(cast(birth_date as date), cast(date_shift_days as int))")) \
    .withColumn("condition_start_date_shifted", expr("date_add(cast(condition_start_date as date), cast(date_shift_days as int))")) \
    .select("patient_id", "ssn_masked", "mrn_masked", "birth_date_shifted", "condition_code", "condition_start_date_shifted")

print("De-Identified Data:")
deid_df.show()


### Step 3: Silver OMOP & DQ/DLQ Routing

In [ ]:
# Filter out bad records for DQ (Circuit Breaker Logic)
valid_df = deid_df.filter(col("patient_id").isNotNull() & (col("condition_code").cast("int") >= 0))
invalid_df = deid_df.filter(col("patient_id").isNull() | (col("condition_code").cast("int") < 0))

print("Silver Valid OMOP Records:")
valid_df.show()

print("DLQ (Invalid) Records:")
invalid_df.show()


### Step 4: Gold Metric (Readmissions via DuckDB)

In [ ]:
con = duckdb.connect(':memory:')

# Register Spark DF as DuckDB View via Pandas
pdf = valid_df.toPandas()
con.register('silver_condition', pdf)

gold_query = """
WITH condition_events AS (
    SELECT 
        patient_id,
        condition_start_date_shifted as admission_date
    FROM silver_condition
    WHERE CAST(condition_code AS INT) = 32020
),
ordered_admissions AS (
    SELECT 
        patient_id,
        admission_date,
        LAG(admission_date) OVER (PARTITION BY patient_id ORDER BY admission_date) as prev_admission_date
    FROM condition_events
)
SELECT 
    patient_id,
    admission_date,
    prev_admission_date,
    DATE_DIFF('day', CAST(prev_admission_date AS DATE), CAST(admission_date AS DATE)) as days_since_last_admission,
    CASE 
        WHEN DATE_DIFF('day', CAST(prev_admission_date AS DATE), CAST(admission_date AS DATE)) <= 30 THEN 1 
        ELSE 0 
    END as is_30_day_readmission
FROM ordered_admissions
WHERE prev_admission_date IS NOT NULL
"""

print("Gold Readmission Metric:")
print(con.execute(gold_query).fetchdf())
